# 8장. LoRA

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


7장에서 몸통을 고정하고 head만 학습한다고 했습니다. 그런데 몸통이 뽑는 특징이 진동 고장 판정에 딱 맞지 않을 수 있습니다. 그렇다고 1억 개 파라미터를 전부 다시 학습하면 데이터도 시간도 부족하고, 사전학습으로 얻은 능력도 망가질 수 있습니다.

**LoRA(Low-Rank Adaptation)**는 몸통은 그대로 두고, 옆에 **아주 작은 보정 장치**를 붙여 그것만 학습하는 방법입니다.

## 8.1 원리

몸통 안의 큰 행렬 `W`(예: 768×768 = 59만 개)를 직접 고치는 대신, 작은 행렬 두 개 `A`(8×768)와 `B`(768×8)를 곱한 값을 더합니다.

```text
원래:   출력 = W · 입력
LoRA:  출력 = W · 입력  +  (alpha / r) · B · A · 입력
              ↑ 고정        ↑ 학습 (A: r×768, B: 768×r)
```

- **r (rank) = 8**: 보정 행렬의 폭. `B·A`는 768×768 크기지만 숫자는 8×768 + 768×8 = 12,288개뿐 (원래의 2%)
- **lora_alpha = 16**: 보정의 세기. 실제로는 `alpha / r = 2`배로 더합니다
- **B는 0으로 시작**: 처음에는 `B·A = 0`이라 사전학습 모델과 똑같이 동작하다가, 학습하며 조금씩 달라집니다
- **lora_dropout = 0.05**: 학습 중 LoRA 입력의 5%를 무작위로 꺼서 외우기를 줄입니다

In [2]:
import torch
a = torch.randn(8, 768); b = torch.zeros(768, 8)            # A는 무작위, B는 0으로 시작
print("B·A 모양:", tuple((b @ a).shape), " 숫자 개수:", a.numel() + b.numel(), "vs W:", 768 * 768)
print("처음 B·A가 전부 0인가?", bool((b @ a == 0).all()), "← 학습 전에는 원래 모델과 같다")

B·A 모양: (768, 768)  숫자 개수: 12288 vs W: 589824
처음 B·A가 전부 0인가? True ← 학습 전에는 원래 모델과 같다


## 8.2 어디에 붙이나: attention의 q, v

Transformer 층마다 **attention**이 있고, 그 안에 행렬 4개(q, k, v, o)가 있습니다.
- **q (query)**: "나는 어떤 정보를 찾나"
- **k (key)**: "나는 어떤 정보를 갖고 있나"
- **v (value)**: "실제로 전달할 내용"
- **o (output)**: 결과를 합치는 행렬

LoRA 논문 이후 **q와 v에만** 붙이는 것이 가볍고 효과적인 기본값으로 널리 쓰입니다. Base는 층 12개 × (q, v) = **24곳**에 붙습니다.

In [3]:
import os, warnings
os.environ["HF_HUB_OFFLINE"] = "1"                 # 인터넷 없이 로컬 모델 폴더만 사용
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"    # Mac: MPS가 모르는 연산은 CPU로
warnings.filterwarnings("ignore")
import torch
from ch09_train_teacher import pick_device
DEVICE = pick_device()
print("장치:", DEVICE)

장치: mps


In [4]:
import ch07_moment as ch07
import ch08_lora as ch08

print("LoRA 설정:", ch08.LORA_CONFIG)
teacher = ch08.add_lora(ch07.load_moment(ROOT / "models/MOMENT-1-base"))
print(teacher.encoder.block[0].layer[0].SelfAttention.q)

LoRA 설정: {'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'target_modules': ['q', 'v'], 'bias': 'none', 'task_type': 'FEATURE_EXTRACTION'}


Loading weights from local directory
lora.Linear(
  (base_layer): Linear(in_features=768, out_features=768, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=768, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=768, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


`q` 행렬이 `base_layer`(원래 W, 고정)와 `lora_A`, `lora_B`(학습) 묶음으로 바뀌었습니다.

## 8.3 파라미터 수 직접 세기

In [5]:
lora = {n: p for n, p in teacher.named_parameters() if "lora_" in n}
head = {n: p for n, p in teacher.named_parameters() if n.startswith("head.")}
total = sum(p.numel() for p in teacher.parameters())
n_lora, n_head = sum(p.numel() for p in lora.values()), sum(p.numel() for p in head.values())
print(f"LoRA 텐서 {len(lora)}개 (A 24 + B 24), 숫자 {n_lora:,}개")
print(f"손 계산: 12층 × 2(q,v) × (8×768 + 768×8) = {12 * 2 * (8 * 768 + 768 * 8):,}")
print(f"head {n_head:,}개 → 학습 대상 {n_lora + n_head:,}개 = 전체 {total:,}개의 {100 * (n_lora + n_head) / total:.2f}%")
print("\n학습 여부 확인:")
for n in list(lora)[:2] + list(head):
    print(f"  {n:58s} {tuple(lora.get(n, head.get(n)).shape)}  학습={lora.get(n, head.get(n)).requires_grad}")

LoRA 텐서 48개 (A 24 + B 24), 숫자 294,912개
손 계산: 12층 × 2(q,v) × (8×768 + 768×8) = 294,912
head 1,538개 → 학습 대상 296,450개 = 전체 109,931,906개의 0.27%

학습 여부 확인:
  encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight (8, 768)  학습=True
  encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight (768, 8)  학습=True
  head.linear.weight                                         (2, 768)  학습=True
  head.linear.bias                                           (2,)  학습=True


## 8.4 gradient checkpointing: 메모리 아끼기

학습할 때는 역전파를 위해 중간 계산 결과를 모두 메모리에 저장해야 합니다. 층이 많으면 메모리가 부족해집니다. **gradient checkpointing**은 중간 결과를 버렸다가, 역전파할 때 **다시 계산**합니다. 메모리를 아끼는 대신 순전파를 한 번 더 하므로 느려집니다.

## 8.5 ⚠ 실제로 겪은 버그: LoRA가 한 번도 학습되지 않았다

이 프로젝트의 첫 Teacher는 20 epoch를 학습했는데 **LoRA B 행렬 24개가 모두 0**이었습니다. LoRA가 한 번도 고쳐지지 않았다는 뜻입니다. 에러도 없었습니다.

**원인**
1. MOMENT는 기본으로 gradient checkpointing을 **reentrant 방식**으로 켭니다.
2. reentrant 방식은 층에 들어오는 **입력 텐서**가 기울기를 필요로 할 때만 역전파 경로를 만듭니다. 층 안의 파라미터(LoRA)는 입력으로 치지 않습니다.
3. 우리는 패치 임베딩을 고정했으므로 인코더 입력은 기울기가 필요 없습니다.
4. → 역전파가 head에서 끊기고 LoRA에는 기울기가 오지 않습니다.
5. optimizer는 기울기가 없는 파라미터를 **조용히 건너뜁니다.** 그래서 에러가 나지 않았습니다.

**해결**: `use_reentrant=False`로 켭니다. 이 방식은 파라미터 기준으로 역전파 경로를 유지합니다.

직접 재현해 봅니다(빠르게 Small로).

In [6]:
for reentrant in (True, False):
    m = ch08.add_lora(ch07.load_moment(ROOT / "models/MOMENT-1-small"), reentrant=reentrant)
    n, got = ch08.one_backward(m)
    print(f"use_reentrant={reentrant!s:5s}: LoRA 파라미터 {n}개 중 기울기를 받은 것 {got}개")

Loading weights from local directory
use_reentrant=True : LoRA 파라미터 32개 중 기울기를 받은 것 0개


Loading weights from local directory
use_reentrant=False: LoRA 파라미터 32개 중 기울기를 받은 것 32개


## 8.6 재발 방지: 기울기 검사

"조용히 실패"하는 것이 가장 위험합니다. 그래서 매 학습 step마다 **학습 대상인데 기울기가 없는 파라미터가 있으면 즉시 멈추는** 검사를 넣습니다.

```python
def validate_gradients(model):
    for name, p in model.named_parameters():
        if p.requires_grad:
            if p.grad is None:
                raise RuntimeError(f"기울기 없음: {name}")
            if not torch.isfinite(p.grad).all():
                raise RuntimeError(f"기울기에 NaN/무한대: {name}")
```

In [7]:
bad = ch08.add_lora(ch07.load_moment(ROOT / "models/MOMENT-1-small"), reentrant=True)
ch08.one_backward(bad)
try:
    ch08.validate_gradients(bad)
except RuntimeError as e:
    print("잡아냄 →", e)

Loading weights from local directory
잡아냄 → 기울기 없음: encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight


> 버그를 고친 뒤 학습이 약 3–4배 느려졌습니다. 이제 역전파가 인코더 전체를 지나가고, checkpointing 때문에 순전파도 한 번 더 하기 때문입니다. 버그가 있을 때의 "빠른 학습"은 사실 LoRA를 건너뛰어서 빨랐던 것입니다.

## 8장 완성 코드: `code/ch08_lora.py`

In [8]:
%run code/ch08_lora.py

Loading weights from local directory
MOMENT-1-small + LoRA(r=8, q/v)
  LoRA 파라미터 114,688개 (손 계산 114,688) + head 1,026개 = 학습 대상 115,714개
  전체 35,453,122개 중 0.33%만 학습

reentrant 버그 재현 (역전파 1번 후 기울기를 받은 LoRA 파라미터 수)


Loading weights from local directory
  use_reentrant=True : 0/32  → 버그! LoRA가 학습되지 않음
    validate_gradients가 잡아냄: 기울기 없음: encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight


Loading weights from local directory
  use_reentrant=False: 32/32  → 정상
    validate_gradients 통과


In [9]:
from IPython.display import Code
Code(filename=str(CODE / "ch08_lora.py"), language="python")

"""8장 완성 코드: MOMENT에 LoRA를 붙이고, 기울기가 LoRA까지 실제로 전달되는지 검사한다.

이 프로젝트에서 실제로 겪은 버그도 재현한다:
    gradient checkpointing을 reentrant 방식으로 켜면 LoRA에 기울기가 전혀 가지 않는다.

실행 (프로젝트 루트에서):
    python docs/manual/code/ch08_lora.py                # Small로 빠르게
    python docs/manual/code/ch08_lora.py --size base    # 실제 Teacher와 같은 Base
"""

import argparse
import warnings
from pathlib import Path

import torch
import torch.nn.functional as F

from ch07_moment import ROOT, load_moment

# 이 프로젝트의 LoRA 설정 (src/bearing_kd/models.py의 LORA_CONFIG)
LORA_CONFIG = {
    "r": 8,                      # 랭크: 보정 행렬의 "폭". 작을수록 가볍다
    "lora_alpha": 16,            # 보정의 세기 = alpha / r = 2배
    "lora_dropout": 0.05,        # 학습 중 LoRA 입력의 5%를 무작위로 끈다 (외우기 방지)
    "target_modules": ["q", "v"],  # attention의 query, value 행렬에만 붙인다
    "bias": "none",
    "task_type": "FEATURE_EXTRACTION",
}


def add_lora(model, config=LORA_CONFIG, reentrant=False):
    """LoRA를 주입하고 학습 대상(LoRA + head)만 requires_grad=True로 둔다."""
    from peft import LoraConfig, inject_adapter_in_model

    model = inject_adapter_in_model(LoraConfig(**config), model)
    # 메모리를 아끼는 gradient checkpointing. 반드시 use_reentrant=False (본문 8.5 참고)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": reentrant})
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("head.") or "lora_" in name
    return model


def expected_lora_params(model, r=8, targets=("q", "v")):
    """LoRA 파라미터 수를 손으로 계산: 행렬 W(out×in)마다 A(r×in) + B(out×r)."""
    total = 0
    for block in model.encoder.block:
        attn = block.layer[0].SelfAttention
        for t in targets:
            base = getattr(attn, t).base_layer if hasattr(getattr(attn, t), "base_layer") else getattr(attn, t)
            out_f, in_f = base.weight.shape
            total += r * in_f + out_f * r
    return total


def validate_gradients(model):
    """학습 대상인데 기울기가 없거나(None) NaN이면 즉시 멈춘다. 조용히 넘어가는 것이 가장 위험하다."""
    for name, p in model.named_parameters():
        if p.requires_grad:
            if p.grad is None:
                raise RuntimeError(f"기울기 없음: {name}")
            if not torch.isfinite(p.grad).all():
                raise RuntimeError(f"기울기에 NaN/무한대: {name}")


def one_backward(model):
    torch.manual_seed(0)
    model.train()
    x, y = torch.randn(4, 1, 512), torch.tensor([0, 1, 0, 1])
    loss = F.cross_entropy(model(x_enc=x, input_mask=torch.ones(4, 512)).logits, y)
    loss.backward()
    lora = [p for n, p in model.named_parameters() if "lora_" in n]
    return len(lora), sum(p.grad is not None for p in lora)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--size", choices=("small", "base"), default="small")
    args = parser.parse_args()
    model_dir = ROOT / f"models/MOMENT-1-{args.size}"

    model = add_lora(load_moment(model_dir))
    lora = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
    head = sum(p.numel() for n, p in model.named_parameters() if n.startswith("head."))
    print(f"MOMENT-1-{args.size} + LoRA(r=8, q/v)")
    print(f"  LoRA 파라미터 {lora:,}개 (손 계산 {expected_lora_params(model):,}) + head {head:,}개 = 학습 대상 {lora + head:,}개")
    total = sum(p.numel() for p in model.parameters())
    print(f"  전체 {total:,}개 중 {100 * (lora + head) / total:.2f}%만 학습")

    print("\nreentrant 버그 재현 (역전파 1번 후 기울기를 받은 LoRA 파라미터 수)")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for reentrant in (True, False):
            m = add_lora(load_moment(model_dir), reentrant=reentrant)
            n, got = one_backward(m)
            status = "정상" if got == n else "버그! LoRA가 학습되지 않음"
            print(f"  use_reentrant={reentrant!s:5s}: {got}/{n}  → {status}")
            try:
                validate_gradients(m)
                print("    validate_gradients 통과")
            except RuntimeError as e:
                print(f"    validate_gradients가 잡아냄: {e}")


if __name__ == "__mai